# Tiled polygonization tutorial

Polygonize a USDA Cropland Data Layer raster in tiles with
`contourrs.shapes_arrow`, merge same-class neighbors across tile
boundaries, and visualize the result.

The sample raster is a 512x512 crop of the 2023 CDL for
Polk County, Iowa (`examples/data/cdl_2023_polk_512.tif`).

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from contourrs import shapes_arrow
from rasterio.windows import Window

## Load the CDL raster

A 512x512 crop of the 2023 USDA Cropland Data Layer (Polk County, IA).
31 land-cover classes, 30m resolution, EPSG:5070.

In [ ]:
raster_path = Path("data/cdl_2023_polk_512.tif")
if not raster_path.exists():
    raster_path = Path("examples/data/cdl_2023_polk_512.tif")

with rasterio.open(raster_path) as src:
    print(f"Size: {src.width}x{src.height}")
    print(f"CRS: {src.crs}")
    print(f"dtype: {src.dtypes[0]}, nodata: {src.nodata}")
    data = src.read(1)
    print(f"Classes: {len(np.unique(data))}")

## Helper: transform tuple

`contourrs` expects a 6-element affine transform tuple `(a, b, c, d, e, f)`.
Rasterio's `Affine` object needs a quick conversion.

In [ ]:
def transform_tuple(t):
    return (t.a, t.b, t.c, t.d, t.e, t.f)

## Polygonize in tiles

Read the raster in 128x128 tiles, polygonize each tile with
`shapes_arrow`, and collect the results into one GeoDataFrame.

In [ ]:
tile_size = 128
connectivity = 4
frames = []

with rasterio.open(raster_path) as src:
    rows = range(0, src.height, tile_size)
    cols = range(0, src.width, tile_size)
    total_tiles = len(rows) * len(cols)
    print(
        f"Raster {src.width}x{src.height}, tile_size={tile_size}, tiles={total_tiles}"
    )

    for row_off in rows:
        for col_off in cols:
            window = Window.from_slices(
                (row_off, min(row_off + tile_size, src.height)),
                (col_off, min(col_off + tile_size, src.width)),
            )
            block = src.read(1, window=window)
            mask = block != 0
            if src.nodata is not None:
                mask &= block != src.nodata
            if not mask.any():
                continue

            table = shapes_arrow(
                block,
                mask=mask,
                connectivity=connectivity,
                transform=transform_tuple(src.window_transform(window)),
            )
            if table.num_rows == 0:
                continue

            tile_gdf = gpd.GeoDataFrame.from_arrow(table)
            tile_gdf = gpd.GeoDataFrame(
                {
                    "value": tile_gdf["value"],
                    "geometry": tile_gdf.geometry,
                },
                geometry="geometry",
                crs=tile_gdf.crs,
            )
            frames.append(tile_gdf)

    crs = src.crs

tiled = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
)
tiled["value"] = tiled["value"].astype("int32")
tiled.set_crs(crs, inplace=True)

print(f"Raw tiled polygons: {len(tiled):,}")

## Merge same-class neighbors

Dissolve polygons that share the same class value, then explode
multipolygons back to individual geometries. This merges
tile-boundary artifacts.

In [ ]:
merged = tiled.dissolve(by="value", as_index=False).explode(
    index_parts=False, ignore_index=True
)
merged = merged[~merged.geometry.is_empty]
merged["value"] = merged["value"].astype("int32")
merged = merged[["value", "geometry"]]

print(f"Merged polygons: {len(merged):,}")
print(f"Classes: {sorted(merged['value'].unique())}")

## Plot: raster vs vector polygons

Side-by-side comparison of the original CDL raster and the
merged vector polygons.

In [ ]:
with rasterio.open(raster_path) as src:
    raster = src.read(1)
    bounds = src.bounds

extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(raster, cmap="tab20", interpolation="nearest", extent=extent)
axes[0].set_title("CDL raster")

merged.plot(
    ax=axes[1],
    column="value",
    cmap="tab20",
    edgecolor="black",
    linewidth=0.03,
    legend=False,
)
axes[1].set_title(f"Merged polygons ({len(merged):,})")

for ax in axes:
    ax.set_axis_off()
    ax.set_xlim(bounds.left, bounds.right)
    ax.set_ylim(bounds.bottom, bounds.top)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()